In [ ]:
from atlite.gis import ExclusionContainer, shape_availability, padded_transform_and_shape
import time
import geopandas as gpd
import numpy as np
import rasterio as rio
from rasterio.enums import Resampling
from numpy import empty
from scipy.ndimage import binary_dilation as dilation
from rasterio.features import geometry_mask

from workflow.scripts.extra_functions import get_bounding

In [ ]:
## method
method = snakemake.params.method # atlite

## input data
corine = snakemake.input.corine
scenario = snakemake.wildcards.scenario
europe_onshore_shp = snakemake.input.europe_onshore_shapefile

# params
# params
corine_codes = snakemake.params.corine_codes
polygon_borders = snakemake.params.polygon_borders
input_scenarios = snakemake.params.input_scenarios["ShadowFlicker"]
target_crs = snakemake.params.target_crs # target EPSG

## output data
output_path = snakemake.output.shadow_flicker_path


In [ ]:
# resolution
res = 100

# borders
[rectx1, rectx2, recty1, recty2] = polygon_borders
polygon_bounds = get_bounding(target_crs,rectx1,rectx2,recty1,recty2)

# buffer
buffer = input_scenarios[scenario]

In [ ]:
europe = (
    gpd
    .read_file(europe_onshore_shp)
    .set_index(["index"])
    .loc[:,['geometry']]
    .to_crs(target_crs)
)

onshore = (
    europe
    .assign(area = 1)
    .dissolve(by='area')
    .reset_index()
)
onshore["area"] = onshore["area"].astype("uint8")

In [ ]:

t=time.time()

if method=="manual":

    dst_transform, dst_shape = padded_transform_and_shape(
        bounds=rio.features.bounds(polygon_bounds), # cutout bounds
        res=res # resolution
    )
    
    src = rio.open(corine)
    raster_corine = src.read(1)
    src_transform = src.transform

    raster_reprojected = empty(dst_shape)
    rio.warp.reproject(
        raster_corine,
        raster_reprojected,
        src_transform=src_transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=target_crs,
        resampling=Resampling.nearest,
        num_threads=2
    )

    # only select some codes in corine
    raster_reprojected = np.where(np.isin(raster_reprojected,corine_codes), 1, 0)

    ## add buffers
    iterations = int(buffer / res) + 1
    raster_buffered = dilation(raster_reprojected, iterations=iterations)

    europe_mask = ~geometry_mask(
        onshore.geometry,
        out_shape=dst_shape,
        transform=dst_transform,
        all_touched=True,
        )

    raster_buffered = europe_mask & raster_buffered

    raster_buffered = raster_buffered.astype("uint8")

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_buffered.shape[1],
        'height': raster_buffered.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': dst_transform,
        'compress': 'lzw'
    }

    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_buffered, 1)
    print(time.time()-t)

1500
75.08163905143738


In [ ]:
t=time.time()

if method=="atlite":
    
    exclusion_container = ExclusionContainer(crs=target_crs)
    # for code in codes_to_buffer:
    exclusion_container.add_raster(corine, codes=corine_codes, buffer=buffer,crs=target_crs)
    # exclusion_container.add_geometry(onshore.geometry, invert=True)
    masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_buffered = (~masked)

    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_geometry(onshore.geometry)
    masked, _ = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_onshore = (~masked)

    raster_buffered = (raster_buffered & raster_onshore).astype("uint8")

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_buffered.shape[1],
        'height': raster_buffered.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': transform,
        'compress': 'lzw'
    }

    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_buffered, 1)
    print(time.time()-t)

1500
113.90638279914856


In [ ]:
if method=="test":
    f = open(output_path, "x")